# AAI614: Data Science & its Applications

*Notebook 3.1: Practice with Data Collections*

<a href="https://colab.research.google.com/github/engmahmoudelhassan-cell/AAI614_Elhassan/blob/main/Week%203/Notebook3.1-M.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Source: Scraping with Python http://shop.oreilly.com/product/0636920034391.do

In [1]:
# Run this cell first. One helper that all the cells below use.
from urllib.request import urlopen, Request
from bs4 import BeautifulSoup
import ssl

ssl._create_default_https_context = ssl._create_unverified_context

HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AAI614 practice'}

def get_soup(url):
    """Open a URL with a real User-Agent (avoids Wikipedia's 403) and return BeautifulSoup."""
    req = Request(url, headers=HEADERS)
    return BeautifulSoup(urlopen(req, timeout=15), 'html.parser')

## List every link on a page

In [2]:
bs = get_soup('https://en.wikipedia.org/wiki/Kevin_Bacon')

for link in bs.find_all('a'):
    if 'href' in link.attrs:
        print(link.attrs['href'])

#bodyContent
/wiki/Main_Page
/wiki/Wikipedia:Contents
/wiki/Portal:Current_events
/wiki/Special:Random
/wiki/Wikipedia:About
//en.wikipedia.org/wiki/Wikipedia:Contact_us
/wiki/Help:Contents
/wiki/Help:Introduction
/wiki/Wikipedia:Community_portal
/wiki/Special:RecentChanges
/wiki/Wikipedia:File_upload_wizard
/wiki/Special:SpecialPages
/w/index.php?title=Special:DownloadAsPdf&page=Kevin_Bacon&action=show-download-screen
/w/index.php?title=Kevin_Bacon&printable=yes
https://commons.wikimedia.org/wiki/Kevin_Bacon
https://www.wikidata.org/wiki/Special:EntityPage/Q3454165
/wiki/Main_Page
/wiki/Special:Search
https://donate.wikimedia.org/?wmf_source=donate&wmf_medium=sidebar&wmf_campaign=en.wikipedia.org&uselang=en
/w/index.php?title=Special:CreateAccount&returnto=Kevin+Bacon
/w/index.php?title=Special:UserLogin&returnto=Kevin+Bacon
https://donate.wikimedia.org/?wmf_source=donate&wmf_medium=sidebar&wmf_campaign=en.wikipedia.org&uselang=en
/w/index.php?title=Special:CreateAccount&returnto=Kevi

## Retrieving Articles Only

In [3]:
import re

bs = get_soup('https://en.wikipedia.org/wiki/Kevin_Bacon')
for link in bs.find('div', {'id': 'bodyContent'}).find_all(
        'a', href=re.compile('^(/wiki/)((?!:).)*$')):
    print(link.attrs['href'])

/wiki/Kevin_Bacon_(disambiguation)
/wiki/Philadelphia
/wiki/Kevin_Bacon_filmography
/wiki/Kyra_Sedgwick
/wiki/Sosie_Bacon
/wiki/Edmund_Bacon_(architect)
/wiki/Michael_Bacon_(composer)
/wiki/Leading_man
/wiki/Golden_Globe_Award
/wiki/Actor_Award
/wiki/National_Lampoon%27s_Animal_House
/wiki/Diner_(1982_film)
/wiki/Footloose_(1984_film)
/wiki/JFK_(film)
/wiki/A_Few_Good_Men
/wiki/Apollo_13_(film)
/wiki/Mystic_River_(film)
/wiki/Frost/Nixon_(film)
/wiki/Friday_the_13th_(1980_film)
/wiki/Tremors_(1990_film)
/wiki/The_River_Wild
/wiki/Balto_(film)
/wiki/The_Woodsman_(2004_film)
/wiki/Crazy,_Stupid,_Love
/wiki/Patriots_Day_(film)
/wiki/Losing_Chase
/wiki/Loverboy_(2005_film)
/wiki/Golden_Globe_Award_for_Best_Actor_%E2%80%93_Miniseries_or_Television_Film
/wiki/Screen_Actors_Guild_Award_for_Outstanding_Performance_by_a_Male_Actor_in_a_Miniseries_or_Television_Movie
/wiki/Michael_Strobl
/wiki/HBO
/wiki/Taking_Chance
/wiki/Fox_Broadcasting_Company
/wiki/The_Following
/wiki/Amazon_Prime_Video
/wi

## Random Walk

In [4]:
import datetime, random, re

random.seed(datetime.datetime.now().timestamp())

def getLinks(articleUrl):
    bs = get_soup(f'https://en.wikipedia.org{articleUrl}')
    return bs.find('div', {'id': 'bodyContent'}).find_all(
        'a', href=re.compile('^(/wiki/)((?!:).)*$'))

MAX_HOPS = 8
links = getLinks('/wiki/Kevin_Bacon')
hops = 0
while len(links) > 0 and hops < MAX_HOPS:
    newArticle = links[random.randint(0, len(links)-1)].attrs['href']
    print(newArticle)
    links = getLinks(newArticle)
    hops += 1
print(f'\nStopped after {hops} hops.')

/wiki/Erd%C5%91s%E2%80%93Bacon_number
/wiki/Twilight_(2008_film)
/wiki/Young_Artist_Awards
/wiki/35th_Young_Artist_Awards
/wiki/Spooksville_(TV_series)
/wiki/35th_Young_Artist_Awards#Outstanding_Young_Ensemble_in_a_TV_Series
/wiki/4th_Youth_in_Film_Awards
/wiki/Archie_Bunker%27s_Place

Stopped after 8 hops.


## Recursively crawling an entire site

In [5]:
import re

pages = set()
MAX_PAGES = 25

def getLinks(pageUrl):
    if len(pages) >= MAX_PAGES:
        return
    bs = get_soup(f'https://en.wikipedia.org{pageUrl}')
    for link in bs.find_all('a', href=re.compile('^(/wiki/)')):
        if 'href' in link.attrs and link.attrs['href'] not in pages:
            if len(pages) >= MAX_PAGES:
                return
            newPage = link.attrs['href']
            print(newPage)
            pages.add(newPage)
            getLinks(newPage)

getLinks('/wiki/Kevin_Bacon')
print(f'\nCollected {len(pages)} pages (limit {MAX_PAGES}).')

/wiki/Main_Page
/wiki/Wikipedia:Contents
/wiki/Portal:Current_events
/wiki/Special:Random
/wiki/Wikipedia:About
/wiki/Help:Contents
/wiki/Help:Introduction
/wiki/Wikipedia:Community_portal
/wiki/Special:RecentChanges
/wiki/Special:Nearby
/wiki/Wikipedia:File_upload_wizard
/wiki/Special:SpecialPages
/wiki/Special:Search
/wiki/Help:Searching
/wiki/Help_talk:Searching
/wiki/Special:WhatLinksHere/Help_talk:Searching
/wiki/Help:What_links_here
/wiki/Help_talk:What_links_here
/wiki/Special:WhatLinksHere/Help_talk:What_links_here
/wiki/Help:What_links_here#Transclusion
/wiki/Special:WhatLinksHere/Help:What_links_here
/wiki/Template_talk:In_use
/wiki/Template:In_use
/wiki/Special:WhatLinksHere/Template:In_use
/wiki/Lithium

Collected 25 pages (limit 25).


## Collecting Data Across an Entire Site

In [6]:
import re

pages = set()
MAX_PAGES = 6

def getLinks(pageUrl):
    if len(pages) >= MAX_PAGES:
        return
    bs = get_soup(f'https://en.wikipedia.org{pageUrl}')
    try:
        print(bs.h1.get_text())
        bodyContent = bs.find('div', {'id': 'bodyContent'}).find_all('p')
        if len(bodyContent):
            print(bodyContent[0].get_text().strip()[:300])
    except AttributeError:
        print('This page is missing something! Continuing.')

    for link in bs.find_all('a', href=re.compile('^(/wiki/)((?!:).)*$')):
        if 'href' in link.attrs and link.attrs['href'] not in pages:
            if len(pages) >= MAX_PAGES:
                return
            newPage = link.attrs['href']
            print('-' * 20)
            print(newPage)
            pages.add(newPage)
            getLinks(newPage)

getLinks('/wiki/General-purpose_programming_language')

General-purpose programming language
In computer software, a general-purpose programming language (GPL) is a programming language for building software in a wide variety of application domains. Conversely, a domain-specific programming language (DSL) is used within a specific area. For example, Python is a GPL, while SQL is a DSL for q
--------------------
/wiki/Main_Page
Main Page
Vatican City's participation at the 2022 Mediterranean Games, held in Oran, Algeria, from 25 June to 6 July 2022, was as guests. It was the Vatican's first appearance in the Mediterranean Games, and its debut in any international multi-sport event. The Vatican's participation was the result of an ag
--------------------
/wiki/Wikipedia
Wikipedia

--------------------
/wiki/English_Wikipedia
English Wikipedia

--------------------
/wiki/Simple_English_Wikipedia
Simple English Wikipedia

--------------------
/wiki/Northern_S%C3%A1mi_Wikipedia
Northern Sámi Wikipedia

--------------------
/wiki/Internet_encyclo

## Collect all External Links from a Site

In [7]:
from urllib.parse import urlparse
import re

def getInternalLinks(bs, url):
    netloc = urlparse(url).netloc
    scheme = urlparse(url).scheme
    internalLinks = set()
    for link in bs.find_all('a'):
        if not link.attrs.get('href'):
            continue
        parsed = urlparse(link.attrs['href'])
        if parsed.netloc == '':
            internalLinks.add(f'{scheme}://{netloc}/{link.attrs["href"].strip("/")}')
        elif parsed.netloc == netloc:
            internalLinks.add(link.attrs['href'])
    return list(internalLinks)

def getExternalLinks(bs, url):
    netloc = urlparse(url).netloc
    externalLinks = set()
    for link in bs.find_all('a'):
        if not link.attrs.get('href'):
            continue
        parsed = urlparse(link.attrs['href'])
        if parsed.netloc != '' and parsed.netloc != netloc:
            externalLinks.add(link.attrs['href'])
    return list(externalLinks)

url = 'https://en.wikipedia.org/wiki/Kevin_Bacon'
bs = get_soup(url)
internal = getInternalLinks(bs, url)
external = getExternalLinks(bs, url)
print('Internal links:', len(internal))
print('External links:', len(external))
print('\nSample external:')
for l in external[:10]:
    print(' ', l)

Internal links: 601
External links: 196

Sample external:
  https://de.wikipedia.org/wiki/Kevin_Bacon
  https://cs.wikipedia.org/wiki/Kevin_Bacon
  https://donate.wikimedia.org/?wmf_source=donate&wmf_medium=sidebar&wmf_campaign=en.wikipedia.org&uselang=en
  https://simple.wikipedia.org/wiki/Kevin_Bacon
  http://www.rogerebert.com/reviews/sleepers-1996
  https://web.archive.org/web/20100722010545/http://heatvision.hollywoodreporter.com/2010/07/winters-bone-star-cast-as-mystique-in-xmen-first-class.html
  https://web.archive.org/web/20080830035710/http://www.time.com/time/magazine/article/0,9171,950019,00.html
  https://azb.wikipedia.org/wiki/%DA%A9%D9%88%DB%8C%D9%86_%D8%A8%DB%8C%DA%A9%D9%86
  http://www.drawtheline.org/watch-stuff
  https://zh-min-nan.wikipedia.org/wiki/Kevin_Bacon


## Crawling across the Internet

In [8]:
import random

def getRandomExternalLink(startingPage):
    bs = get_soup(startingPage)
    externalLinks = getExternalLinks(bs, startingPage)
    if not externalLinks:
        internalLinks = getInternalLinks(bs, startingPage)
        if not internalLinks:
            return None
        return getRandomExternalLink(random.choice(internalLinks))
    return random.choice(externalLinks)

def followExternalOnly(startingSite, max_hops=5):
    site = startingSite
    for i in range(max_hops):
        try:
            site = getRandomExternalLink(site)
        except Exception as e:
            print(f'Stopped at hop {i}: {e}')
            return
        if not site:
            print('No external link found, stopping.')
            return
        print(f'Hop {i+1}: {site}')

followExternalOnly('https://www.oreilly.com/')

Hop 1: https://learning.oreilly.com/search/?query=author%3A%22Arianne%20Dee%22&extended_publisher_data=true&highlight=true&include_assessments=false&include_case_studies=true&include_courses=true&include_playlists=true&include_collections=true&include_notebooks=true&include_sandboxes=true&include_scenarios=true&is_academic_institution_account=false&source=suggestion&sort=date_added&facet_json=true&json_facets=true&page=0&include_facets=false
Stopped at hop 1: HTTP Error 403: Forbidden


---
# Experiments: different HTML tags and CSS attributes

### 1. Section headings — selecting by tag name
The `<h2>` tags are the section titles. Grabbing them gives a quick outline of the page without reading any of it.

In [9]:
bs = get_soup('https://en.wikipedia.org/wiki/Kevin_Bacon')
headings = [h.get_text(strip=True) for h in bs.find_all(['h2', 'h3'])]
print(len(headings), 'headings found')
for h in headings:
    print(' -', h)

17 headings found
 - Contents
 - Early life and education
 - Acting career
 - Early work
 - 1980s
 - 1990s
 - 2000s
 - 2010s
 - Other ventures
 - Six Degrees of Kevin Bacon
 - Personal life
 - Accolades
 - Awards and nominations
 - Other honors
 - See also
 - References
 - External links


### 2. The infobox — selecting by CSS class
The box on the right of a Wikipedia article is a `<table class="infobox">`. Reading its `<th>`/`<td>` rows turns it into clean key/value facts, which is the kind of structured data worth scraping.

In [10]:
import pandas as pd

bs = get_soup('https://en.wikipedia.org/wiki/Kevin_Bacon')
infobox = bs.find('table', class_='infobox')

facts = {}
for row in infobox.find_all('tr'):
    th, td = row.find('th'), row.find('td')
    if th and td:
        facts[th.get_text(' ', strip=True)] = td.get_text(' ', strip=True)

df = pd.DataFrame(list(facts.items()), columns=['Field', 'Value'])
df

,Field,Value
0,Born,"Kevin Norwood Bacon ( 1958-07-08 ) July 8, 195..."
1,Occupations,Actor musician
2,Years active,1977–present
3,Works,Filmography
4,Spouse,Kyra Sedgwick ​ ( m. 1988) ​
5,Children,"2, including Sosie Bacon"
6,Father,Edmund Bacon
7,Relatives,Michael Bacon (brother)
8,Website,baconbros .com


### 3. CSS selectors with select()
`select()` takes the same selectors you'd write in CSS, which is often shorter than chaining `find`/`find_all`. `#id`, `.class`, `tag.class`, and attribute filters like `a[href^="/wiki/"]` all work.

In [11]:
bs = get_soup('https://en.wikipedia.org/wiki/Kevin_Bacon')

print('infobox rows          :', len(bs.select('table.infobox tr')))
print('body paragraphs       :', len(bs.select('div#bodyContent p')))
print('internal wiki links   :', len(bs.select('div#bodyContent a[href^="/wiki/"]')))
print('external (https) links:', len(bs.select('div#bodyContent a[href^="https"]')))
# same article-only filter as before, but as one CSS line:
print('first 5 article links :')
for a in bs.select('div#bodyContent a[href^="/wiki/"]')[:5]:
    print('   ', a.get('href'))

infobox rows          : 11
body paragraphs       : 36
internal wiki links   : 560
external (https) links: 77
first 5 article links :
    /wiki/Wikipedia:Protection_policy#semi
    /wiki/Kevin_Bacon_(disambiguation)
    /wiki/File:Kevin_Bacon_The_Best_You_Can-55_(cropped).jpg
    /wiki/Philadelphia
    /wiki/Kevin_Bacon_filmography


### 4. What is the page actually made of:  counting tags
Counting every tag name shows where the content lives. On an article most of the weight is in `<a>`, `<li>` and `<span>`, which is why naive link scraping returns so much noise.

In [12]:
from collections import Counter

bs = get_soup('https://en.wikipedia.org/wiki/Kevin_Bacon')
counts = Counter(tag.name for tag in bs.find_all(True))
for name, n in counts.most_common(12):
    print(f'{name:8} {n}')

a        1010
span     904
li       472
div      244
td       194
i        186
sup      109
link     107
b        90
tr       75
ul       70
cite     68


### 5. Citations — selecting by class to get sources
References sit in `<cite class="citation ...">` tags. Pulling them is a fast way to harvest a page's source list.

In [13]:
bs = get_soup('https://en.wikipedia.org/wiki/Kevin_Bacon')
cites = bs.select('cite.citation')
print(len(cites), 'citations on the page. First 10:')
for c in cites[:10]:
    print(' -', c.get_text(' ', strip=True)[:120])

68 citations on the page. First 10:
 - Roberts, Gary Boyd. "Ten Further Hollywood Figures (or Groups Thereof)" . New England Historic Genealogical Society. Arc
 - "Kevin Bacon" . Biography.com . Retrieved July 21, 2012 .
 - Porter, Rick (October 27, 2022). " 'City on a Hill' Canceled After Three Seasons on Showtime" . The Hollywood Reporter .
 - "Hollywood Walk of Fame - Kevin Bacon" . Hollywood Walk of Fame . September 30, 2003 . Retrieved May 9, 2017 .
 - O'Reilly, Laura (October 1, 2012). "EE unveils 'six degrees of Bacon' launch ads" . Marketing Week . Retrieved January 4
 - "Kevin Bacon: 6 Things You Didn't Know" . biography.com. Archived from the original on April 3, 2019 . Retrieved October
 - "4 Stars From Philly To Hollywood" . CBS Philadelphia . CBS News. October 10, 2010 . Retrieved June 9, 2019 .
 - "ABOUT KEVIN BACON" . yahoo movies . Retrieved July 21, 2012 .
 - "Kevin Bacon biography" . biography channel. Archived from the original on October 16, 2014 . Retrieved October

### 6. The filmography table straight into pandas
For a real `<table>`, `pandas.read_html` does the parsing for you and hands back a DataFrame. Faster than walking rows by hand when the table is well formed.

In [16]:
import pandas as pd
from io import StringIO

bs = get_soup('https://en.wikipedia.org/wiki/Kevin_Bacon')
tables = pd.read_html(StringIO(str(bs)))
print(len(tables), 'tables parsed from the page.')
# show the first reasonably sized table
for t in tables:
    if t.shape[0] >= 3 and t.shape[1] >= 2:
        display(t.head(10))
        break

11 tables parsed from the page.


,Kevin Bacon,Kevin Bacon.1
0,Bacon in 2025,Bacon in 2025
1,Born,"Kevin Norwood Bacon July 8, 1958 (age 67) Phil..."
2,Occupations,Actor musician
3,Years active,1977–present
4,Works,Filmography
5,Spouse,Kyra Sedgwick ​(m. 1988)​
6,Children,"2, including Sosie Bacon"
7,Father,Edmund Bacon
8,Relatives,Michael Bacon (brother)
9,Website,baconbros.com
